In [ ]:
#@title 🎮 Paso 1: ¡probemos primero un juego de Family Computer! { display-mode: "form" }
#@markdown Presioná el botón de reproducción (▶) y el programa en lenguaje C se compilará automáticamente; debajo aparecerá la pantalla del juego.
#@markdown #@markdown Cuando aparezca la pantalla, **hacé clic una vez dentro del área del juego** para activar el sonido y luego probá los controles con el teclado.


# 0. Configuración del objetivo
# ────【Cambiá esto para alternar entre actividades (sin extensión)】────
TARGET = "game"
# ────────────────────────────────────────────────────────


# 1. Instalación del entorno de desarrollo (compilador cc65) para todas las personas
!apt-get install cc65


# 2. [Opcional] Definición de la función para conectar Google Drive # (si aparece un error con una cuenta escolar u otra cuenta administrada, ejecutá manualmente `mountdrive()` al final)
import os
def mountdrive():
  from google.colab import drive
  drive.mount('/content/drive')
  target_dir = "/content/drive/MyDrive"
  os.makedirs(target_dir, exist_ok=True)
  %cd $target_dir
# ─── Si querés usar Drive, quitá el "#" de la siguiente línea y ejecutala ───
# mountdrive()


# 3. Solo la primera vez: descargar y descomprimir el entorno (NESLab) en el Drive de cada persona
# (si la carpeta ya existe, conviene omitir este paso automáticamente)
# Si Drive no está conectado, volver a /content; si ya está conectado, mantener el %cd anterior
if 'target_dir' not in locals():
 %cd /content
if not os.path.exists("NESLab"):
 !git clone https://github.com/ip-arch/NESLab.git
# 4. Moverse con seguridad al directorio correcto de NESLab


if os.path.exists("NESLab"):
 %cd NESLab
 !pwd
!ls
# 5. Crear game.nes


import os
import base64
from IPython.display import HTML


c_file = f"{TARGET}.c"
nes_file = f"{TARGET}.nes"


print(f"Directorio actual: {os.getcwd()}")


# 1. Verificar si existe el archivo C especificado
if not os.path.exists(c_file):
    print(f"❌ Error: no se encontró {c_file}. Verificá el nombre del archivo.")
else:
    # 2. Eliminar artefactos anteriores y compilar solo el archivo de actividad especificado
    print(f"📦 Compilando {c_file}...")
    !make clean
    !make {nes_file} M65CDIR=/usr


    # 3. Si la compilación fue exitosa, cargar el archivo NES e iniciar el emulador
    if os.path.exists(nes_file):
        print(f"🚀 ¡{nes_file} se inició correctamente! Hacé clic en la pantalla de abajo para habilitar el sonido.")

        with open(nes_file, "rb") as f:
            nes_bytes = f.read()
        nes_base64 = base64.b64encode(nes_bytes).decode('utf-8')


        # HTML del emulador con soporte de audio (el archivo cargado cambia dinámicamente)
        html_code = f"""
        <div id="nes-container" style="text-align: center; background: #222; padding: 15px; border-radius: 8px; color: white; font-family: sans-serif; max-width: 540px; margin: 0 auto;">
            <h4 style="margin-top: 0; color: #ff4a4a;">🔊 Emulador NES [{nes_file}]</h4>
            <canvas id="nes-canvas" width="256" height="240" style="width: 512px; height: 480px; background: black; border: 4px solid #444; border-radius: 4px;"></canvas>
            <p style="font-size: 0.85rem; color: #ccc; margin-top: 10px; line-height: 1.4;">
                【Importante】 <b>¡Hacé clic una vez dentro de la pantalla para activar el sonido!</b><br>
                <b>Flechas</b>: mover | <b>Tecla Z</b>: botón B | <b>Tecla X</b>: botón A<br>
                <b>Tecla Enter</b>: START | <b>Barra espaciadora</b>: SELECT
            </p>
        </div>


        <script src="https://cdnjs.cloudflare.com/ajax/libs/jsnes/1.2.1/jsnes.min.js"></script>
        <script>
        (function() {{
            var canvas = document.getElementById('nes-canvas');
            var ctx = canvas.getContext('2d');
            var imageData = ctx.getImageData(0, 0, 256, 240);


            var AUDIO_BUFFER_SIZE = 1024;
            var SAMPLE_COUNT = 44100 * 2;
            var audioBuffer = new Float32Array(SAMPLE_COUNT);
            var bufferReadIdx = 0;
            var bufferWriteIdx = 0;
            var audioCtx = null;
            var scriptNode = null;


            function initAudio() {{
                if (audioCtx) return;
                audioCtx = new (window.AudioContext || window.webkitAudioContext)({{ sampleRate: 44100 }});
                scriptNode = audioCtx.createScriptProcessor(AUDIO_BUFFER_SIZE, 0, 2);

                scriptNode.onaudioprocess = function(e) {{
                    var outputLeft = e.outputBuffer.getChannelData(0);
                    var outputRight = e.outputBuffer.getChannelData(1);
                    var available = (bufferWriteIdx - bufferReadIdx + SAMPLE_COUNT) % SAMPLE_COUNT;

                    if (available > 8192) {{
                        bufferReadIdx = (bufferWriteIdx - 4096 + SAMPLE_COUNT) % SAMPLE_COUNT;
                    }}


                    for (var i = 0; i < AUDIO_BUFFER_SIZE; i++) {{
                        if (bufferReadIdx === bufferWriteIdx) {{
                            outputLeft[i] = 0; outputRight[i] = 0;
                        }} else {{
                            outputLeft[i] = audioBuffer[bufferReadIdx];
                            bufferReadIdx = (bufferReadIdx + 1) % SAMPLE_COUNT;
                            outputRight[i] = audioBuffer[bufferReadIdx];
                            bufferReadIdx = (bufferReadIdx + 1) % SAMPLE_COUNT;
                        }}
                    }}
                }};
                scriptNode.connect(audioCtx.destination);
            }}


            var nes = new jsnes.NES({{
                onFrame: function(frameBuffer) {{
                    var d = imageData.data;
                    for (var i = 0; i < frameBuffer.length; i++) {{
                        var p = frameBuffer[i];
                        var idx = i * 4;
                        d[idx]     = (p >> 16) & 0xff;
                        d[idx + 1] = (p >> 8) & 0xff;
                        d[idx + 2] = p & 0xff;
                        d[idx + 3] = 0xff;
                    }}
                    ctx.putImageData(imageData, 0, 0);
                }},
                onAudioSample: function(left, right) {{
                    if (!audioCtx) return;
                    audioBuffer[bufferWriteIdx] = left;
                    bufferWriteIdx = (bufferWriteIdx + 1) % SAMPLE_COUNT;
                    audioBuffer[bufferWriteIdx] = right;
                    bufferWriteIdx = (bufferWriteIdx + 1) % SAMPLE_COUNT;
                }}
            }});


            var romData = atob("{nes_base64}");
            nes.loadROM(romData);


            var keyboard = function(callback, event) {{
                var player = 1;
                switch(event.keyCode) {{
                    case 38: callback(player, jsnes.Controller.BUTTON_UP); event.preventDefault(); break;
                    case 40: callback(player, jsnes.Controller.BUTTON_DOWN); event.preventDefault(); break;
                    case 37: callback(player, jsnes.Controller.BUTTON_LEFT); event.preventDefault(); break;
                    case 39: callback(player, jsnes.Controller.BUTTON_RIGHT); event.preventDefault(); break;
                    case 88: callback(player, jsnes.Controller.BUTTON_A); event.preventDefault(); break;
                    case 90: callback(player, jsnes.Controller.BUTTON_B); event.preventDefault(); break;
                    case 13: callback(player, jsnes.Controller.BUTTON_START); event.preventDefault(); break;
                    case 32: callback(player, jsnes.Controller.BUTTON_SELECT); event.preventDefault(); break;
                }}
            }};


            document.addEventListener('keydown', function(e) {{ keyboard(nes.buttonDown, e); }});
            document.addEventListener('keyup', function(e) {{ keyboard(nes.buttonUp, e); }});


            document.getElementById('nes-container').addEventListener('click', function() {{
                initAudio();
                if (audioCtx && audioCtx.state === 'suspended') {{ audioCtx.resume(); }}
                document.querySelector('#nes-container h4').style.color = '#4ae2ff';
                document.querySelector('#nes-container h4').innerText = '🎮 Emulador NES [{nes_file}] (sonido ACTIVO)';
            }});


            function step() {{ nes.frame(); requestAnimationFrame(step); }}
            requestAnimationFrame(step);
        }})();
        </script>
        """
        display(HTML(html_code))
    else:
        print(f"❌ Error: no se pudo generar {nes_file}. Revisá la sintaxis del código C u otros posibles problemas.")







